# Chapter 20 — Context Windows and Truncation

**Book alignment:** Debugging AI From First Principles, Chapter 20

**Question this notebook isolates:** Short split-shipment queries pass; the same question
with full ticket history fails. Does a length ledger plus three truncation-side probes —
**shorten**, **reorder**, **budget** — separate a tail/middle **cut** (H1) from a mid-window
**burial** (H2) from output-budget **starvation** (H3)?

In [ ]:
EFFECTIVE_IN = 8000          # measured from deployment config, NOT the advertised window
RESERVED_OUT = 1000

SEGMENTS = [                 # (name, tokens, priority)  priority: history > docs
    ("system",   200, 3),
    ("history", 4000, 2),
    ("doc 1.0-general",   1500, 1),
    ("doc 3.4-shipping",  1500, 1),
    ("doc 4.2-exception", 1500, 1),   # the decisive section
    ("doc 5.1-misc",      1500, 1),
    ("question",  100, 3),
]

def assemble(segments, effective_in=EFFECTIVE_IN):
    """Block-level policy: keep system+question+history; the docs block is all-or-nothing."""
    fixed = [s for s in segments if s[2] >= 2]
    docs  = [s for s in segments if s[2] == 1]
    used  = sum(t for _, t, _ in fixed)
    if used + sum(t for _, t, _ in docs) <= effective_in:
        kept = fixed + docs
    else:
        kept = fixed                          # docs don't all fit -> drop the whole block
    return {name for name, _, _ in kept}

## 1. Ledger the lengths — the cut is arithmetic

In [ ]:
total = sum(t for _, t, _ in SEGMENTS)
kept = assemble(SEGMENTS)
print(f"assembled tokens {total}  vs effective input {EFFECTIVE_IN}  (+ {RESERVED_OUT} reserved out)")
print("kept segments:", sorted(kept))
assert total > EFFECTIVE_IN
assert "doc 4.2-exception" not in kept          # dropped with the docs block
print("policy dropped the entire docs block -> 4.2 is physically ABSENT at generation time")

## 2. Three probes, one intervention each, forecasts pre-written

In [ ]:
def passes(segments, effective_in=EFFECTIVE_IN, out_budget=RESERVED_OUT):
    kept = assemble(segments, effective_in)
    present = "doc 4.2-exception" in kept
    finished = out_budget >= 600                 # a full citation needs ~600 output tokens
    return present and finished

# shorten: strip low-priority history so the docs block fits (changes total tokens)
shorten = [s for s in SEGMENTS if s[0] != "history"]
# reorder: move 4.2 to the front of the docs block - SAME token multiset, still over budget
reorder = [SEGMENTS[i] for i in (0, 1, 4, 2, 3, 5, 6)]
# budget: same input, double the output reservation
print("shorten :", passes(shorten))
print("reorder :", passes(reorder))
print("budget  :", passes(SEGMENTS, out_budget=2000))
assert passes(shorten) is True                   # H1 forecast: only shorten flips
assert passes(reorder) is False                  # H2 exonerated: order doesn't add it back
assert passes(SEGMENTS, out_budget=2000) is False # H3 exonerated: nothing was starved
print("\nshorten flips, reorder + budget do not -> H1: the section was CUT, not buried or starved")

## 3. The length sweep — a separate effect, held distinct

In [ ]:
# even with 4.2 forced present and uncut, very long contexts degrade use (a RULER-style effect)
def use_rate(context_tokens):
    return max(0.4, 1.0 - 0.00004 * context_tokens)   # illustrative, per-fixture, NOT a law

for ctx in (2000, 6000, 12000, 20000):
    print(f"context {ctx:>6} tok  ->  4.2-use rate {use_rate(ctx):.2f}")
assert use_rate(2000) > use_rate(20000) + 0.3
print("a length-sensitive failure with 4.2 present-and-uncut is NOT truncation - measure per fixture")

## 4. Pin the policy as a ledger assertion

In [ ]:
LEDGER_ASSERTION = {
    "per_segment_caps": {"history": 1500, "docs_block": 6000},
    "non_truncatable": ["doc 4.2-exception", "question"],
    "truncation_markers": "required - emit <dropped: name, tokens> per removed segment",
    "ci_gate": "assert assembled_tokens + reserved_out <= effective_in",
}
for k, v in LEDGER_ASSERTION.items():
    print(f"{k}: {v}")
# cap history so the whole docs block fits under budget -> 4.2 survives:
capped = [(n, min(t, 1500) if n == "history" else t, p) for n, t, p in SEGMENTS]
assert sum(t for _, t, _ in capped) <= EFFECTIVE_IN
assert "doc 4.2-exception" in assemble(capped)

## What we earned

The window is a ledger with a truncation policy: tokens in per segment, tokens allowed,
tokens reserved, and a rule for which end dies first. The arithmetic named the dropped row —
a block-level policy discarded the entire docs block, so section 4.2 was physically absent
at generation. Three probes kept the diagnosis single-variable: only **shorten** (which
frees budget) flipped the failure; **reorder** (same tokens) and **budget** (more output
room) did not — a **cut**, not a burial or a starved finish. The length sweep is a distinct
effect, measured per fixture, never cited as a law.

**Notebook 21 / Chapter 21** treats the variance that survives a balanced ledger: sampling
as part of the program, and the behavioral distribution as the object.